# 🏦 Santander Dev Week 2023 — ETL com Python & IA Generativa

> **Notebook exploratório** — espelha o desafio original da DIO, adaptado para a stack Python moderna.
> Para execução completa e automatizada, utilize o pipeline modular via `python main.py` ou `make run-etl`.

---

## Contexto

A tarefa original era consumir uma API Java (Spring Boot + Railway) para obter dados de clientes bancários,
gerar mensagens de marketing personalizadas via IA Generativa (GPT-4) e persistir os resultados de volta na API.

**Adaptações realizadas nesta versão:**

| Componente | Original | Esta versão |
|---|---|---|
| API Backend | Java + Spring Boot + Railway *(descontinuado)* | FastAPI + SQLite (local) |
| IA Generativa | OpenAI GPT-4 (pago) | Groq + Llama-3 70B (gratuito) |
| Execução | Notebook único | Pipeline modular + Notebook |

---

## ⚙️ Configuração

Antes de executar este notebook, certifique-se de que:
1. A API local está no ar: `make run-api` (ou `uvicorn api.main:app --reload`)
2. O banco foi populado: `make seed`
3. O arquivo `.env` existe com a `GROQ_API_KEY` preenchida

In [ ]:
# Instala dependências (caso necessário)
%pip install fastapi uvicorn sqlalchemy pydantic pandas requests groq python-dotenv --quiet

In [ ]:
import json
import os
import sys

import pandas as pd
import requests
from dotenv import load_dotenv

# Garante que os módulos do projeto sejam encontrados
sys.path.insert(0, os.path.abspath('..'))
load_dotenv('../.env')

API_URL = os.getenv('API_URL', 'http://localhost:8000')
print(f'API_URL configurada: {API_URL}')

---
## 📥 Etapa 1 — Extract

Lê os IDs do arquivo `SDW2023.csv` e busca os dados de cada usuário via `GET /users/{id}`.

In [ ]:
# Carrega o CSV com os IDs de usuário
df = pd.read_csv('../data/SDW2023.csv')
user_ids = df['UserID'].tolist()

print(f'IDs carregados: {user_ids}')
df

In [ ]:
def get_user(user_id: int) -> dict | None:
    """Busca um usuário na API local pelo ID."""
    response = requests.get(f'{API_URL}/users/{user_id}')
    if response.status_code == 200:
        return response.json()
    print(f'  ⚠️  Usuário {user_id} não encontrado (status {response.status_code})')
    return None

# Extrai todos os usuários
users = []
for uid in user_ids:
    user = get_user(uid)
    if user:
        user.setdefault('news', [])
        users.append(user)
        print(f'  ✅ Extraído: {user["name"]} (id={user["id"]})')

print(f'\n{len(users)} usuário(s) extraídos com sucesso.')

In [ ]:
# Visualiza os dados extraídos
print(json.dumps(users, indent=2, ensure_ascii=False))

---
## 🤖 Etapa 2 — Transform

Gera mensagens de marketing personalizadas para cada usuário usando o modelo **Llama-3 70B** via **Groq API**.

In [ ]:
from groq import Groq

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
if not GROQ_API_KEY:
    raise EnvironmentError('GROQ_API_KEY não encontrada. Verifique o arquivo .env')

client = Groq(api_key=GROQ_API_KEY)

ICON_URL = (
    'https://digitalinnovationone.github.io'
    '/santander-dev-week-2023-api/icons/credit.svg'
)

SYSTEM_PROMPT = (
    'Você é um especialista em marketing bancário do Santander Brasil. '
    'Seu tom é próximo, motivador e financeiramente educativo. '
    'Escreva sempre em português brasileiro.'
)

def generate_news(user: dict) -> str:
    """Gera mensagem personalizada sobre investimentos via Llama-3."""
    first_name = user['name'].split()[0]
    balance    = user.get('account', {}).get('balance', 0.0)

    completion = client.chat.completions.create(
        model='llama3-70b-8192',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': (
                    f'Crie uma mensagem curta e personalizada para {first_name}, '
                    f'cliente do Santander, destacando a importância de investir '
                    f'o dinheiro parado na conta (saldo atual: R$ {balance:.2f}). '
                    f'Máximo de 110 caracteres, use o primeiro nome do cliente.'
                )
            }
        ],
        max_tokens=80,
        temperature=0.85,
    )
    return completion.choices[0].message.content.strip().strip('"')

print('Gerando mensagens...')

In [ ]:
# Gera e anexa as mensagens a cada usuário
for user in users:
    news_text = generate_news(user)
    user['news'].append({'icon': ICON_URL, 'description': news_text})
    print(f'  💬 {user["name"]}: "{news_text}"')

---
## 📤 Etapa 3 — Load

Persiste as mensagens geradas de volta na API via `PUT /users/{id}`.

In [ ]:
def update_user(user: dict) -> bool:
    """Envia as novas 'news' via PUT /users/{id}."""
    payload = {
        'news': [
            {'icon': n['icon'], 'description': n['description']}
            for n in user.get('news', [])
        ]
    }
    response = requests.put(f'{API_URL}/users/{user["id"]}', json=payload)
    return response.status_code == 200

# Atualiza todos os usuários
for user in users:
    success = update_user(user)
    status  = '✅' if success else '❌'
    print(f'  {status} {user["name"]} atualizado? {success}')

---
## 💾 Resultado Final

In [ ]:
# Salva resultado em JSON
output_path = '../data/output_users.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(users, f, ensure_ascii=False, indent=2)

print(f'Resultado salvo em {output_path}')
print(json.dumps(users, indent=2, ensure_ascii=False))

---

## ✅ Conclusão

O pipeline ETL foi executado com sucesso:

1. **Extract** — IDs lidos do CSV → dados buscados via `GET /users/{id}` na API local (FastAPI + SQLite)
2. **Transform** — Mensagens personalizadas geradas pelo modelo **Llama-3 70B** (Groq API) em português
3. **Load** — Mensagens persistidas via `PUT /users/{id}` na API local

Para o pipeline completo e automatizado com logging estruturado, execute:
```bash
make run-etl
```